In [1]:
# import torch
# from collections import OrderedDict

# cp=torch.load("/workspace/tmp/csm/csm_1b.pt", map_location='cpu')

# kept_weights = []

# for (k,v) in cp.items():
#     if 'decoder.' in k or 'audio_embeddings.' in k or 'projection.' in k or 'codebook0_head' in k or 'audio_head' in k:
#         kept_weights.append((k,v))

# torch.save(OrderedDict(kept_weights), "/workspace/tmp/v4-crash-course/csm-depth-decoder-weights.pt")

In [5]:
# Code from https://github.com/SesameAILabs/csm/blob/main/models.py
import torch
import torch.nn as nn
import torchtune
from torchtune.models import llama3_2

def llama3_2_100M() -> torchtune.modules.transformer.TransformerDecoder:
    return llama3_2.llama3_2(
        vocab_size=128_256,
        num_layers=4,
        num_heads=8,
        num_kv_heads=2,
        embed_dim=1024,
        max_seq_len=2048,
        intermediate_dim=8192,
        attn_dropout=0.0,
        norm_eps=1e-5,
        rope_base=500_000,
        scale_factor=32,
    )

def _prepare_transformer(model):
    embed_dim = model.tok_embeddings.embedding_dim
    # No embedding lookup (pass in pre looked up embeddings)
    model.tok_embeddings = nn.Identity()
    # No lm_head
    model.output = nn.Identity()
    return model, embed_dim

class CSMDepthDecoder(nn.Module):
    def __init__(
        self,
        csm_audio_vocab_size=2051,  # For some reason CSM vocab size is 2048+3
        csm_audio_num_codebooks=32,
        csm_backbone_dim=2048,
        # Rime depth decoder
        qwen_backbone_dim=1536,
        rime_num_codebooks=12
    ):
        super().__init__()

        self.csm_audio_vocab_size = csm_audio_vocab_size
        self.rime_num_codebooks = rime_num_codebooks

        # Just load all relevant weights in original shapes for now (can prune shapes later)
        self.decoder, decoder_dim = _prepare_transformer(llama3_2_100M())
        
        self.audio_embeddings = nn.Embedding(csm_audio_vocab_size * csm_audio_num_codebooks, csm_backbone_dim)
        
        self.projection = nn.Linear(csm_backbone_dim, decoder_dim, bias=False)
        self.codebook0_head = nn.Linear(csm_backbone_dim, csm_audio_vocab_size, bias=False)
        self.audio_head = nn.Parameter(torch.empty(csm_audio_num_codebooks - 1, decoder_dim, 2051))

        # Load relevant pre-trained weights
        cp = torch.load("/workspace/tmp/v4-crash-course/csm-depth-decoder-weights.pt", map_location='cpu')
        self.load_state_dict(cp)

        # Random init projection from Qwen hidden dim to CSM embedding dim
        self.qwenH_to_csmH = nn.Linear(qwen_backbone_dim, csm_backbone_dim, bias=False)

    def set_csm_trainable(self, trainable: bool):
        
        for module in [ self.decoder, self.audio_embeddings, self.projection, self.codebook0_head ]:
            for p in module.parameters():
                p.requires_grad = trainable

        # Parameter: audio_head
        self.audio_head.requires_grad = trainable
    
    def forward(self, backbone_hidden_states, mimi_targets, freeze_csm_weights=True):
        
        # Case in case backbone_hidden_states in bfloat16
        backbone_hidden_states = backbone_hidden_states.to(
            dtype=self.qwenH_to_csmH.weight.dtype # 
        )

        # Up-project Qwen to match CSM hidden states [B, 1, 1536] -> [B, 1, 2048]
        csmH = self.qwenH_to_csmH(backbone_hidden_states)

        # Input embeds: [B, C, 2048] from [B, 1, 2048] prefix onto [B, C-1, 2048] (all but last codebook)
        input_embeds = torch.cat(
            [ csmH ] + 
            [ self.audio_embeddings(mimi_targets[:, i] + (i * self.csm_audio_vocab_size)).unsqueeze(1) for i in range(self.rime_num_codebooks - 1) ],
            dim=1
        )

        # Following CSM, down-project input_embeds [B, C, 2048] -> [B, C, 1024] right before input to decoder
        decoder_h = self.decoder(self.projection(input_embeds))

        # Codebook 0 targets are predicted directly from hidden state
        # Codebook 1-N targets are predicted from Llama hidden state
        logits_stacked = torch.stack([ self.codebook0_head(csmH[:,0,:]) ] + \
            [ torch.mm(decoder_h[:, i, :], self.audio_head[i]) for i in range(self.rime_num_codebooks - 1) ],
             dim=1
        )

        loss = torch.nn.functional.cross_entropy(
            logits_stacked.reshape(-1, self.csm_audio_vocab_size), # (B*C, V)
            mimi_targets.reshape(-1),                              # (B*C,)
            reduction="mean"
        )

        return loss
        
model = CSMDepthDecoder()

In [6]:
# [B, 1, H_qwen]
backbone_hidden_states = torch.rand((4, 1, 1536))

# Mimi targets
mimi_targets = torch.randint(0, 2048, (4, 12))

In [7]:
model(backbone_hidden_states, mimi_targets)

tensor(10.5344, grad_fn=<NllLossBackward0>)